# Maya v2 — continuous streaming with in-flight MFC commands

This notebook uses `maya_v2.py`, which keeps one persistent USB Serial
connection to the Teensy and runs the sensor-packet reader in a background
thread. That lets us switch MFCs *while* the sensor stream is running,
without stopping and restarting the stream.

**Safe to call during streaming:** `MFCControl.set_flow`, `MFCControl.blink`,
`PCF8575Control_.send_state` (binary command, no USB text response).

**NOT safe during streaming:** `MFCControl.read_registers` — the firmware
writes `DATA:...` on the USB Serial and would corrupt the binary packet
stream. `maya_v2.MFCControl` refuses it while streaming.

In [ ]:
import os
import time
import pickle
import numpy as np
import matplotlib.pyplot as plt

from maya_v2 import SerialCommunication, PCF8575Control_, MFCControl

In [ ]:
# --- Serial / hardware configuration ---
PORT = "/dev/ttyACM0"   # adjust to your Teensy's serial port
BAUDRATE = 115200
FREQUENCY = 1           # sampling frequency (Hz) set on the Teensy

# DAC configuration
dac1_value = 800
dac2_value = 200

# PCF8575 configuration
PCF_flags = [True] * 16
PCF_frequencies = ["0"] * 16

sensor_id = ["U1-TGS2602", "U2-TGS2610-D00", "U3-SP3S-AQ2", "U4-GSBT11-DXX",
             "U8-TGS2600", "U9-TGS2603", "U10-TGS2630", "U13-TGS2612-D00",
             "U14-TGS2620", "U15-MG-812", "U16-TGS-3830", "U19-TGS1820",
             "U20-TGS2611-E00", "U21-TGS2616-C00", "U22-WSP2110", "U25-TGS-3870",
             "U7-BME680"]

In [ ]:
# Open ONE persistent serial connection. All commands and the streaming
# reader share this same `SerialCommunication` object.
serial_comm = SerialCommunication(PORT, BAUDRATE)

# Push initial DAC + sampling-frequency config to the Teensy
serial_comm.send_message(f"{dac1_value},{dac2_value},{FREQUENCY}")

# Initial PCF8575 state (all sensors ON)
PCF_control = PCF8575Control_(serial_comm, flags=PCF_flags, frequencies=PCF_frequencies)
PCF_control.send_state()

# Zero all MFCs before we begin
MFC_control = MFCControl(serial_comm)
for addr in (1, 2, 3):
    MFC_control.set_flow(addr, 0.0)

## Sanity-check experiment — 30 s sequence

Quick test to confirm that MFC commands sent during a live stream are honoured and show up in the sensor trace at the right times.

Timeline (wall-clock relative to `start_streaming`):

- `t =  0 s` — streaming starts, all MFCs closed
- `t = 10 s` — MFC 2 opened to flow = 1.0
- `t = 20 s` — MFC 2 closed
- `t = 30 s` — streaming stops

Every MFC event is logged with a timestamp in `times_seq` so the sequence can be aligned with the sensor trace afterwards.

In [ ]:
# --- 30 s sanity-check experiment (no CSV, in-memory only) ---
#   t =  0 s : streaming starts, all MFCs closed
#   t = 10 s : MFC 2 -> flow 1.0
#   t = 20 s : MFC 2 -> flow 0.0
#   t = 30 s : streaming stops

# Make sure everything is closed before we start
for addr in (1, 2, 3):
    MFC_control.set_flow(addr, 0.0)

times_seq = []   # (unix_t, asctime, event, mfc_addr, flow)

# Stream to memory only; buffer big enough for the whole run.
serial_comm.start_streaming(save_csv=False, buffer_size=10_000)
try:
    t0 = time.time()
    times_seq.append((time.time(), time.asctime(), "stream_start", 0, 0.0))

    time.sleep(10.0 - (time.time() - t0))
    MFC_control.set_flow(2, 1.0)
    times_seq.append((time.time(), time.asctime(), "on", 2, 1.0))

    time.sleep(20.0 - (time.time() - t0))
    MFC_control.set_flow(2, 0.0)
    times_seq.append((time.time(), time.asctime(), "off", 2, 0.0))

    time.sleep(30.0 - (time.time() - t0))
    times_seq.append((time.time(), time.asctime(), "stream_end", 0, 0.0))
    print(f"Experiment wall-time: {time.time() - t0:.2f} s")
finally:
    packets = serial_comm.get_latest_packets()
    serial_comm.stop_streaming()

# --- Parse in-memory packets and plot ---
sensor_data = []
times = []
for row in packets:
    times.append(row[0])
    values = []
    for i in range(17):
        b1 = int(row[2 * i + 1]); b2 = int(row[2 * i + 2])
        values.append(int.from_bytes([b1, b2], byteorder="little"))
    sensor_data.append(values)
sensor_data = np.array(sensor_data)
print(f"Captured {sensor_data.shape[0]} packets × {sensor_data.shape[1]} channels")

if len(sensor_data) > 0:
    fig, ax = plt.subplots(4, 4, figsize=(16, 12), sharex=True)
    for i in range(4):
        for j in range(4):
            idx = i * 4 + j
            ax[i, j].plot(sensor_data[:, idx])
            ax[i, j].set_title(sensor_id[idx], fontsize=9)
            ax[i, j].set_xticks([])
    fig.suptitle("Sanity check — MFC2 open at t=10s, closed at t=20s")
    plt.tight_layout()
    plt.show()

## Main experiment — 36 min recording with MFC2/3 mixture + MFC1 pulses

Total recording: **36 min = 2160 s**.

**MFC 2 & 3 mixture** — every 5 s the (flow2, flow3) setpoint is drawn uniformly at random from:

```
(0.00, 1.00), (0.25, 0.75), (0.50, 0.50), (0.75, 0.25), (1.00, 0.00)
```

Consecutive periods may repeat the same configuration (independent sampling). 2160 / 5 = **432 periods**.

**MFC 1 pulses** — 30 pulses, one per minute, with durations drawn from
`{1 s × 10, 2 s × 10, 4 s × 10}` shuffled randomly:

- 0 – 300 s (5 min): baseline, MFC1 closed
- 300 – 2100 s (30 min): one pulse at the start of each minute (MFC1 = 1.0 for `duration` s, then 0.0)
- 2100 – 2160 s (1 min): buffer after the last pulse

All event timestamps (config changes, pulse on/off) are recorded in `times_seq` and pickled alongside the CSV.

In [ ]:
import random

# --- Experiment parameters ---
T_TOTAL        = 36 * 60     # 2160 s
T_BASELINE     = 5 * 60      # 300 s -- no MFC1 pulses before this
T_BUFFER       = 1 * 60      # 60 s  -- no MFC1 pulses after the last one
CONFIG_PERIOD  = 5           # s between MFC2/MFC3 setpoint changes
PULSE_INTERVAL = 60          # s between successive MFC1 pulse starts
PULSE_OFFSET   = 2           # s offset of each pulse within its minute,
                             # so pulse edges don't coincide with config changes
PULSE_COUNT    = {1: 10, 2: 10, 4: 10}

CONFIGS = [(0.00, 1.00), (0.25, 0.75), (0.50, 0.50), (0.75, 0.25), (1.00, 0.00)]

filename = "data/v2_mix_pulse_36min.csv"

# --- Build the schedule ---
# MFC2/3 config sequence: one config per 5 s period, sampled independently.
n_periods = T_TOTAL // CONFIG_PERIOD
config_sequence = [random.choice(CONFIGS) for _ in range(n_periods)]

# MFC1 pulse durations: 10 of each, shuffled.
pulse_durations = sum(([d] * n for d, n in PULSE_COUNT.items()), [])
random.shuffle(pulse_durations)

assert len(pulse_durations) == 30
n_pulse_slots = (T_TOTAL - T_BASELINE - T_BUFFER) // PULSE_INTERVAL
assert n_pulse_slots == 30, f"Expected 30 pulse slots, got {n_pulse_slots}"

# Build (t, action, payload) events.
events = []
for i, cfg in enumerate(config_sequence):
    events.append((i * CONFIG_PERIOD, "config", cfg))
for k, dur in enumerate(pulse_durations):
    t_on  = T_BASELINE + k * PULSE_INTERVAL + PULSE_OFFSET
    t_off = t_on + dur
    events.append((t_on,  "pulse_on",  dur))
    events.append((t_off, "pulse_off", dur))
events.sort(key=lambda e: e[0])

print(f"Periods: {n_periods}, pulses: {len(pulse_durations)} "
      f"({PULSE_COUNT}), total duration: {T_TOTAL}s")
print(f"First 5 pulse durations in order: {pulse_durations[:5]}")

# --- Run the experiment ---
if os.path.exists(filename):
    os.remove(filename)
os.makedirs(os.path.dirname(filename), exist_ok=True)

# Make sure all MFCs are closed before starting.
for addr in (1, 2, 3):
    MFC_control.set_flow(addr, 0.0)

times_seq = []  # (unix_t, t_target_s, action, payload)

serial_comm.start_streaming(filename=filename, save_csv=True)
try:
    t0 = time.time()
    times_seq.append((time.time(), 0.0, "stream_start", None))

    for t_target, action, payload in events:
        now = time.time() - t0
        sleep_for = t_target - now
        if sleep_for > 0:
            time.sleep(sleep_for)

        if action == "config":
            f2, f3 = payload
            MFC_control.set_flow(2, f2)
            MFC_control.set_flow(3, f3)
        elif action == "pulse_on":
            MFC_control.set_flow(1, 1.0)
        elif action == "pulse_off":
            MFC_control.set_flow(1, 0.0)

        times_seq.append((time.time(), float(t_target), action, payload))

    # Tail: run out the clock to the full T_TOTAL.
    remaining = T_TOTAL - (time.time() - t0)
    if remaining > 0:
        time.sleep(remaining)

    times_seq.append((time.time(), float(T_TOTAL), "stream_end", None))
    print(f"Experiment wall-time: {time.time() - t0:.2f} s "
          f"(target {T_TOTAL} s)")
finally:
    # Always close MFC1 and stop the stream, even on interrupt.
    try:
        MFC_control.set_flow(1, 0.0)
    except Exception:
        pass
    serial_comm.stop_streaming()

with open(f"{filename}_sequence.pkl", "wb") as f:
    pickle.dump(
        {
            "events": times_seq,
            "config_sequence": config_sequence,
            "pulse_durations": pulse_durations,
            "params": {
                "T_TOTAL": T_TOTAL,
                "T_BASELINE": T_BASELINE,
                "T_BUFFER": T_BUFFER,
                "CONFIG_PERIOD": CONFIG_PERIOD,
                "PULSE_INTERVAL": PULSE_INTERVAL,
                "PULSE_OFFSET": PULSE_OFFSET,
                "CONFIGS": CONFIGS,
            },
        },
        f,
    )
print(f"Sequence log saved to {filename}_sequence.pkl")

## Cleanup

In [ ]:
# Zero the MFCs and close the serial port when you're done.
for addr in (1, 2, 3):
    MFC_control.set_flow(addr, 0.0)
serial_comm.close()

## Main experiment — load and plot sensor traces

Reads the CSV written by the main experiment cell above. Pulse windows (MFC1 on) are shaded on each sensor trace so the pulse response is visible at a glance.

In [ ]:
import csv
from datetime import datetime

# Re-load the main experiment's CSV and sequence log.
with open(f"{filename}_sequence.pkl", "rb") as f:
    seq = pickle.load(f)
events = seq["events"]

sensor_data = []
times = []
with open(filename, "r") as f:
    reader = csv.reader(f)
    for row in reader:
        if not row or row[0] == "Timestamp":
            continue
        times.append(row[0])
        values = []
        for i in range(17):
            b1 = int(row[2 * i + 1]); b2 = int(row[2 * i + 2])
            values.append(int.from_bytes([b1, b2], byteorder="little"))
        sensor_data.append(values)
sensor_data = np.array(sensor_data)
print(f"Loaded {sensor_data.shape[0]} packets × {sensor_data.shape[1]} channels")

# Convert packet timestamps to seconds since stream start.
t_stream_start = events[0][0]  # unix_t of stream_start event
t_seconds = np.array([
    (datetime.strptime(ts, "%Y-%m-%d %H:%M:%S.%f").timestamp() - t_stream_start)
    for ts in times
])

# Extract pulse on/off windows for shading.
pulse_windows = []
t_on = None
for unix_t, t_target, action, payload in events:
    t_rel = unix_t - t_stream_start
    if action == "pulse_on":
        t_on = t_rel
    elif action == "pulse_off" and t_on is not None:
        pulse_windows.append((t_on, t_rel, payload))
        t_on = None

In [ ]:
if len(sensor_data) > 0:
    # Drop the BME680 channel (index 16) from the stacked plot, same as the
    # original notebook which only shows the 16 gas sensors.
    data_T = sensor_data[:, :16].T  # shape (16, n_packets)

    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111)

    for i, row in enumerate(data_T):
        baseline = np.mean(row[:10])
        signal = (np.array(row) - baseline) / 65536.
        if i == 0:
            ax.plot(t_seconds, signal, lw=0.6, label=sensor_id[i])
        else:
            ax.plot(t_seconds, signal * 6 + 2 * i, lw=0.6, label=sensor_id[i])

    # Shade MFC1 pulse windows.
    y_lo, y_hi = -1, 2 * len(data_T) + 1
    for t_on, t_off, dur in pulse_windows:
        ax.fill([t_on, t_on, t_off, t_off], [y_lo, y_hi, y_hi, y_lo],
                color="gray", alpha=0.25)

    ax.set_yticks(2 * np.arange(len(data_T)), labels=sensor_id[:len(data_T)])
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(left=False, labelleft=True)
    ax.set_xlim([t_seconds[0], t_seconds[-1]])
    ax.set_ylim([y_lo, y_hi])
    ax.set_xlabel("t (s)")
    fig.suptitle(f"Main experiment — {filename} "
                 f"({sensor_data.shape[0]} packets, {len(pulse_windows)} pulses)")
    plt.tight_layout()
    plt.show()